# 🧾 Results & Conclusions — every number, with its boundary attached

Notebook 13 showed you how the evidence was *earned*: seven experiments, each answering one
skeptic's question, run by `e2e.experiments` against the real stack. This chapter is the
verdict. You will recompute **every headline number yourself** from the committed dataset
(`e2e/runs/eval/`), attach the honest caveat each one carries, and end at the conclusions a
thesis defense would stand on — including the negative findings, stated plainly instead of
hidden.

**What you'll be able to do after this notebook**

- read a measured number like an examiner: median, spread, `n` — and refuse any number that
  arrives without its simulation boundary
- recompute the headline results live (89 ms lifecycle, 86 ns predicate, lag ≈ poll,
  268k/447k gas, 12/12 attacks rejected, 12/12 decisions, +69 ms trustlessness) and pin
  each one with an `assert` against `summary.json`
- say which numbers would survive a move to a public testnet unchanged, and which must be
  remeasured
- derive the feasibility verdict from the data and defend every line of it
- turn the honest negative findings into scoped future work

**You need:** notebooks 00–13 (13 for what each experiment *is*, 08 for the predicate,
06 for gas). **No extra infrastructure** — the dataset is committed in the repo, and
matplotlib ships with the `demo` dependency group.

*Estimated time: 60–75 minutes.*

## 1. How to read a measured number honestly

Before any result: how do we report a measurement without lying by accident? Latency-like
data is *skewed* — most runs are quick and near-identical, a few are stragglers (a garbage
collection, a cold cache, a busy CPU). The **mean** (sum ÷ count) is dragged by a single
straggler; the **median** (sort the values, take the middle one) is what a *typical* run
actually saw. That is why every latency number in `docs/09-evaluation.md` is a median.

A median alone is still not honest. The house rules, used all chapter:

1. **median + `[min–max]`** — the spread shows the worst case instead of averaging it away
   (at n=20 a "p95" is essentially the max, so docs/09 refuses to cite one);
2. **`n` disclosed** — twenty quiet runs are evidence, two are an anecdote;
3. **no number without its simulation boundary** — this one gets its own cell below.

Watch the toy first: one straggler poisons the mean, and the median doesn't move.

In [ ]:
import statistics

quiet = [86, 88, 89, 91]              # four ordinary runs, in milliseconds
runs = quiet + [2400]                 # ...plus one straggler (a 2.4 s hiccup)

print(f"runs             → {runs}")
print(f"mean             → {statistics.mean(runs):7.1f} ms   ← poisoned by ONE bad run")
print(f"median           → {statistics.median(runs):7.1f} ms   ← what a typical run saw")
print(f"spread [min–max] → [{min(runs)}, {max(runs)}] ms       ← the straggler is DISCLOSED, not hidden")

assert statistics.median(runs) == 89      # the middle of the sorted five — unmoved
assert statistics.mean(runs) > 500        # the mean now describes NO run at all
print("✓ median robust, mean poisoned — that is why docs/09 reports medians")

**✏️ Your turn 1.1 — the median by hand.** With an *even* number of values there is no
single middle element: the median is the average of the two middle ones. The four values
below are real `det`-mode lifecycle times (in ms, rounded) from the dataset you're about to
load. Sort them, average the two middle values by hand, and check against
`statistics.median`. Success: your hand-computed value equals the library's exactly.

In [ ]:
vals = [68, 129, 89, 88]              # real det e2e times in ms: the min, the max, two middles

ordered = sorted(vals)
print("sorted →", ordered)

hand_median = 0.0                     # TODO: average the two middle values of `ordered` by hand
print(f"your median → {hand_median}    library → {statistics.median(vals)}")

# TODO: un-comment once hand_median is computed:
# assert hand_median == statistics.median(vals)

<details><summary>✅ Solution 1.1 — peek only after trying</summary>

```python
hand_median = (ordered[1] + ordered[2]) / 2   # (88 + 89) / 2 = 88.5
assert hand_median == statistics.median(vals)
```

With even `n` the median sits *between* two real observations — which is fine: it still
can't be dragged by the 129 the way a mean would be.
</details>

House rule 3 is the one this chapter exists to drill: **no number without its simulation
boundary.** Notebook 13 named the five things the testbed simulates; each caps what a
specific number may claim. Here they are again — as *data*, because we will attach them to
every result. When a section ends with `boundary(1, 2)`, that is the measurement's fine
print, printed right next to the number instead of buried in a footnote.

In [ ]:
BOUNDARIES = {
    1: "instant-mine chain — Anvil mines the moment a tx arrives; every chain wait is a LOWER bound",
    2: "in-process, co-located — no A2A/HTTP hop in the timed path; a transport-free lower bound",
    3: "ADR-006 datapath carve-out — 'enforced' = config committed via gNMI AND read back, not packets shaped",
    4: "n=20, sequential, one machine, one run, warm — typical cost, NOT tails or throughput",
    5: "one LLM (Qwen3-4B), one deployment, one session — judgment numbers describe THIS deployment",
}

def boundary(*ids: int) -> None:
    """The fine print, attached to the number it caps (house rule 3)."""
    for i in ids:
        print(f"  ⚠ boundary {i}: {BOUNDARIES[i]}")

assert len(BOUNDARIES) == 5
print("the five simulation boundaries (named in 13), now executable:")
boundary(*BOUNDARIES)

## 2. The dataset on the table

The harness wrote one JSONL file per experiment (one JSON object per line — the same
crash-safe, `cat`-able format as the dashboard's event log) plus `summary.json`, the folded
statistics. All of it is **committed in the repo as evidence**, so this chapter needs no
lab, no chain, no LLM endpoint — just files.

House discipline for the whole chapter: `summary.json` is the *claim*; the JSONL lines are
the *evidence*. Every headline number gets recomputed from raw lines, then asserted against
the summary. Look for: seven files, 80/80 lifecycles, zero failures.

In [ ]:
import json
from pathlib import Path

import a2a_interfaces.fixtures as fx

ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())
EVAL = ROOT / "e2e" / "runs" / "eval"

def read(name: str) -> list[dict]:
    return [json.loads(line) for line in (EVAL / name).read_text().splitlines() if line.strip()]

LAT    = read("latency.jsonl")        # E1 — 80 phase-timed lifecycles (also carries E2 lag + E3 gas)
EXPIRY = read("expiry.jsonl")         # E2 — chain-time expiry → device deconfig
BASE   = read("baseline.jsonl")       # E6 — bare netctl, no trust machinery
ADV    = read("adversarial.jsonl")    # E4 — the attack matrix
LLMJ   = read("llm.jsonl")            # E5 — the two judgment slots
PRED   = read("predicate.jsonl")      # E7 — the isolated predicate
SWEEP  = read("revlag_sweep.jsonl")   # E9 — revocation lag vs watcher poll
SUMMARY = json.loads((EVAL / "summary.json").read_text())

for name, rows in [("latency", LAT), ("expiry", EXPIRY), ("baseline", BASE), ("adversarial", ADV),
                   ("llm", LLMJ), ("predicate", PRED), ("revlag_sweep", SWEEP)]:
    print(f"  {name:14} → {len(rows):3} rows")
print(f"  lifecycles     → {SUMMARY['runs_ok']}/{SUMMARY['runs_total']} ok · failures: {SUMMARY['failures']}")

assert SUMMARY["runs_ok"] == SUMMARY["runs_total"] == 80 and SUMMARY["failures"] == []
assert int(fx.PRICE_10_TOK) == 10 * 10**18   # the canonical 10 TOK — the list price these runs paid
print("✓ 80/80 lifecycles, zero failures — the evidence is on the table")

Figures sit *next to* numbers in this chapter, never instead of them: every chart is
followed by the printed values it draws. The palette is the course's trust-domain identity
— purple `agent`, amber `chain`, cyan `controller`, green `network` — checked for
color-vision-deficiency separation, with white gaps between stacked segments so shape
carries each boundary even where color can't. `spread()` below packages section 1's honest
triple (median, `[min–max]`, `n`) so every number arrives fully dressed.

In [ ]:
import statistics

import matplotlib as mpl
import matplotlib.pyplot as plt

INK, MUTED, GRID = "#1a1f2e", "#6b7688", "#e6e9f0"
C = dict(agent="#8B7BF0", sign="#A99BF4", chain="#E0A93B",
         controller="#3BC5D6", network="#38C475", alert="#F0556B")
mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "font.size": 11,
    "axes.edgecolor": GRID, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110,
})

def spread(values: list[float]) -> dict:
    """median + [min–max] + n — section 1's honest triple, as a dict."""
    return {"n": len(values), "median": statistics.median(values),
            "min": min(values), "max": max(values)}

def show_ms(label: str, s: dict) -> None:
    print(f"{label:40} median {s['median'] * 1000:7.1f} ms  "
          f"[{s['min'] * 1000:.0f}–{s['max'] * 1000:.0f}]  (n={s['n']})")

print("plot style + spread() ready — every claim below gets computed, then asserted")

**✏️ Your turn 2.1 — know your rows.** Before computing anything, know the shape of one
raw record. Predict (write it as the comment) how many of the 80 lifecycle rows ran in
`llm` mode, then peek at the keys one row carries. Success: your prediction matches, and
you can name where the per-phase timings live inside a row.

In [ ]:
print("keys of one lifecycle row →", sorted(LAT[0].keys()))
print("phases measured           →", sorted(LAT[0]["phases"].keys()))

# My prediction: ____ of the 80 rows are llm-mode
n_llm = sum(1 for r in LAT if r["mode"] == "llm")
print("llm-mode rows →", n_llm)

# TODO: un-comment once you've predicted:
# assert n_llm == 40                  # 20 runs × 2 services — and the same again for det

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
assert n_llm == 40    # n=20 per mode×service: 20 × 2 services = 40 per mode
```

Each row is one full lifecycle: `mode`, `service`, `ok`, a `gas` dict, and `phases` — the
per-phase stopwatch dict every section below mines.
</details>

## 3. E1 — where the time goes

One lifecycle = negotiate → sign → settle → challenge → activate. The harness timed each
phase; `e2e_request_to_enforced_s` sums them — request to *config committed on srl1*
("enforced" in the ADR-006 sense, boundary 3; revocation happens afterwards and is measured
separately in section 5). Two modes ran: `det` (fixed canonical price, no LLM anywhere) and
`llm` (the two real judgment slots against the deployed Qwen3-4B).

First the headline, computed from raw rows — then proven identical to `summary.json`'s
claim, bit for bit, because it is the same formula over the same data. Look for **~89 ms**
det vs **~3.27 s** llm.

In [ ]:
det_e2e = [r["phases"]["e2e_request_to_enforced_s"] for r in LAT if r["mode"] == "det" and r["ok"]]
llm_e2e = [r["phases"]["e2e_request_to_enforced_s"] for r in LAT if r["mode"] == "llm" and r["ok"]]

show_ms("det e2e (pooled, both services)", spread(det_e2e))
show_ms("llm e2e (pooled, both services)", spread(llm_e2e))

assert len(det_e2e) == len(llm_e2e) == 40
assert 0.085 < statistics.median(det_e2e) < 0.095        # ~89 ms
assert 3.0 < statistics.median(llm_e2e) < 3.5            # ~3.27 s

for mode in ("det", "llm"):                              # our medians ARE the summary's —
    for svc in ("bandwidth", "telemetry"):               # same data, same formula, bit for bit
        ours = statistics.median([r["phases"]["e2e_request_to_enforced_s"] for r in LAT
                                  if r["mode"] == mode and r["service"] == svc and r["ok"]])
        claimed = SUMMARY["latency"][f"{mode}/{svc}"]["e2e_request_to_enforced_s"]["median"]
        assert ours == claimed, (mode, svc)
print("✓ recomputed all four group medians — every one matches summary.json exactly")

Now *where* those milliseconds live. Five segments per lifecycle, colored by trust domain:
**negotiate** (the two LLM judgment slots — quote + decide), **sign** (EIP-712 offer +
EIP-191 proof, from 07), **settle** (the atomic swap on chain, 06), **controller compute**
(challenge + predicate + translation + chain reads — `activate` minus the gNMI write), and
**actuate** (the gNMI Set itself, 09). In `det` mode negotiate is ~0 by construction: there
is no deterministic consumer graph in production — the det path takes the canonical list
price, so the det→llm gap *is* the full cost of judgment on this model.

Look for: the det row is a sliver. That sliver is the entire deterministic machinery this
architecture contributes.

In [ ]:
SEG_COLORS = {"negotiate (LLM)": C["agent"], "sign (EIP-712/191)": C["sign"],
              "settle (chain)": C["chain"], "controller compute": C["controller"],
              "actuate (gNMI→srl1)": C["network"]}

def segments(r: dict) -> dict:
    p = r["phases"]
    return {"negotiate (LLM)": p.get("quote_s", 0.0) + p.get("decide_s", 0.0),
            "sign (EIP-712/191)": p["sign_offer_s"] + p["sign_proof_s"],
            "settle (chain)": p["settle_s"],
            "controller compute": p["challenge_s"] + max(0.0, p["activate_s"] - p["gnmi_apply_s"]),
            "actuate (gNMI→srl1)": p["gnmi_apply_s"]}

med_seg = {m: {s: statistics.median([segments(r)[s] for r in LAT if r["mode"] == m and r["ok"]])
               for s in SEG_COLORS} for m in ("det", "llm")}

fig, ax = plt.subplots(figsize=(9.5, 2.7))
for yi, m in enumerate(("det", "llm")):
    left = 0.0
    for seg, col in SEG_COLORS.items():
        v = med_seg[m][seg] * 1000
        ax.barh(yi, v, left=left, height=0.55, color=col, edgecolor="white",
                linewidth=2, label=seg if yi == 0 else None)
        left += v
    ax.text(left + 45, yi, f"≈{left:,.0f} ms", va="center", color=MUTED, fontsize=10)
ax.set_yticks([0, 1], ["det", "llm"]); ax.invert_yaxis(); ax.set_xlim(0, 3700)
ax.set_xlabel("milliseconds (median per phase, pooled n=40 per mode)")
ax.set_title("E1 — the deterministic machinery is the sliver; the LLM is the bar", color=INK, pad=8)
ax.legend(loc="upper center", fontsize=8.5, ncols=3, framealpha=1)
plt.tight_layout(); plt.show()

print(f"{'segment':26}{'det':>10}{'llm':>10}")
for seg in SEG_COLORS:
    print(f"  {seg:24}{med_seg['det'][seg] * 1000:8.1f}ms{med_seg['llm'][seg] * 1000:8.1f}ms")
print("(segment medians; medians don't sum exactly to the e2e median)")

The purple segment is the point of the whole figure. Rule 1 confines LLM judgment to two
slots (Ada's decide, Bell's quote — 10) and keeps it **outside** the enforcement path:
nothing the LLM says can configure a router; it can only decide whether to *shop*. The data
now prices that design decision: the two judgment calls are ~97 % of the llm-mode wall
clock — and removing them entirely (`det` mode) leaves enforcement intact at 89 ms. The
slow part is swappable *policy*; the security-bearing part is the fast sliver. Rule 1,
paying off as a number.

In [ ]:
share = statistics.median([(r["phases"]["quote_s"] + r["phases"]["decide_s"])
                           / r["phases"]["e2e_request_to_enforced_s"]
                           for r in LAT if r["mode"] == "llm" and r["ok"]])
print(f"judgment share of llm-mode wall clock → {share:.1%}   ← rule 1, as a number")
print(f"enforcement path without judgment     → {statistics.median(det_e2e) * 1000:.0f} ms")

assert 0.90 < share < 0.99
boundary(1, 2, 3, 4)

**✏️ Your turn 3.1 — your median vs the summary's.** Recompute the `llm/telemetry`
end-to-end median from the raw rows and check it against the summary's claim, exactly the
way the chapter did for all four groups. Success: your `assert` passes with `==` — not
"approximately" — because evidence and claim are the same computation.

In [ ]:
rows = [r for r in LAT if r["mode"] == "llm" and r["service"] == "telemetry" and r["ok"]]
print(f"{len(rows)} llm/telemetry lifecycles loaded")

my_median = 0.0                       # TODO: median of e2e_request_to_enforced_s over `rows`
claimed = SUMMARY["latency"]["llm/telemetry"]["e2e_request_to_enforced_s"]["median"]
print(f"yours → {my_median:.4f} s    summary claims → {claimed:.4f} s")

# TODO: un-comment when computed:
# assert my_median == claimed

<details><summary>✅ Solution 3.1 — peek only after trying</summary>

```python
my_median = statistics.median([r["phases"]["e2e_request_to_enforced_s"] for r in rows])
assert my_median == claimed
```

`summary.json` holds no information the JSONL doesn't — it is a fold of the same lines, so
recomputing it is exact, not approximate.
</details>

## 4. E7 — the 86-nanosecond bouncer (the sharpest number)

In 08 you proved `controller.domain.predicate` pure: no I/O imports (rule 4), same inputs →
same answer, every time. Purity buys two things *at once*. **Auditability** — six readable
checks anyone can verify by reading. And, now measured, **speed** — a pure function can be
honestly micro-benchmarked (`timeit`, 200,000 calls per outcome: nothing to wait on, no
state to reset between calls), and it runs in *nanoseconds* (1 ns = a billionth of a
second). This isolates the security decision from the chain reads and the gNMI write that
`activate` bundles around it — 13 walked the harness code; here we read its verdict.

In [ ]:
ns = {r["outcome"]: r["ns_per_call"] for r in PRED}
denials = {k: v for k, v in ns.items() if k != "allow"}

for outcome, v in ns.items():
    note = "   ← the full six-check allow path" if outcome == "allow" else ""
    print(f"  {outcome:14} {v:6.1f} ns/call{note}")

assert 80 < ns["allow"] < 95                        # ~86 ns for a full allow
assert all(45 < v < 90 for v in denials.values())   # every denial 51–83 ns
assert max(denials.values()) < ns["allow"]          # every denial exits EARLIER than allow

llm_decide = statistics.median([r["phases"]["decide_s"] for r in LAT if r["mode"] == "llm" and r["ok"]])
ratio = llm_decide / (ns["allow"] * 1e-9)
print(f"\none LLM decide ≈ {ratio:,.0f} predicate calls   ← seven orders of magnitude apart")
assert ratio > 1e7

The bar chart's *order* tells its own story. The predicate runs its checks in a fixed
sequence, and a denial returns the moment a check fails — so the earlier the check, the
cheaper the denial. `E_NOT_OWNER` (the first check) is cheapest at ~51 ns; `E_CONFLICT`
(the last) costs almost the full allow path. The bars mirror the check order you walked in
08 — the *shape* of the data confirming the *structure* of the code.

In [ ]:
from matplotlib.patches import Patch

names = list(ns)
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(names, [ns[k] for k in names], edgecolor="white",
        color=[C["controller"] if k == "allow" else C["alert"] for k in names])
ax.invert_yaxis()
ax.set_xlabel("nanoseconds per call (timeit, 200,000 calls per outcome)")
ax.set_title("E7 — authorization costs nanoseconds; denials exit early, and cheaper", color=INK, pad=8)
ax.set_xlim(0, 120)              # headroom so the legend never covers a bar
ax.legend(handles=[Patch(color=C["controller"], label="allow (all six checks)"),
                   Patch(color=C["alert"], label="denials (early exit)")],
          loc="upper right", fontsize=9, framealpha=1)
plt.tight_layout(); plt.show()

denial_order = [r["ns_per_call"] for r in PRED if r["outcome"] != "allow"]
assert denial_order == sorted(denial_order)   # file order = check order = cost order
print("✓ denial cost rises with check position — the code's structure, visible in the data")

**✏️ Your turn 4.1 — how many bounces per device write?** The det-mode gNMI apply
(`gnmi_apply_s`) is the physical floor of actuation. Compute how many predicate calls fit
inside ONE median device write — predict the order of magnitude first (thousands?
millions?). Mind the units: `ns["allow"]` is in *nano*seconds, `gnmi_med` in seconds.
Success: a six-figure count, and "authorization is computationally free" now has a number
behind it.

In [ ]:
gnmi_med = statistics.median([r["phases"]["gnmi_apply_s"] for r in LAT if r["mode"] == "det" and r["ok"]])
print(f"one median gNMI apply → {gnmi_med * 1000:.1f} ms")

# My prediction: ____ predicate calls fit in one device write (order of magnitude)
calls_per_write = 0                   # TODO: gnmi_med ÷ one allow (convert ns → seconds!)

print(f"predicate calls per device write → {calls_per_write:,.0f}")
# TODO: un-comment once computed:
# assert 100_000 < calls_per_write < 500_000

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
calls_per_write = gnmi_med / (ns["allow"] * 1e-9)   # ≈ 257,000
assert 100_000 < calls_per_write < 500_000
```

A quarter of a million authorization decisions fit inside one router write — the trust
boundary is *not* where this system's time goes.
</details>

## 5. E2 — the ticket physically governs the wire

The thesis line under test: the ERC-721 entitlement is *authorization, not paperwork* —
when the chain says stop, the router config goes away. Two stop signals exist, and 12
performed both. **Expiry** first: chain time passes `end_time`, and the ExpiryTimer's tick
re-checks `chain_time()` (ADR-004 — the OS timer only schedules the wake-up, the chain
decides) then tears down with one synchronous gNMI delete. No transaction, no watcher — the
deadline was written on the ticket all along. Look for ~73 ms: essentially a single device
round-trip.

In [ ]:
ticks = [r["tick_to_deconfig_s"] for r in EXPIRY if r["ok"]]
show_ms("expiry: tick → device deconfigured", spread(ticks))

assert len(ticks) == 10
assert 0.060 < statistics.median(ticks) < 0.090      # ~73 ms — one gNMI round-trip
print("✓ chain-time expiry becomes device deconfiguration in ~73 ms")

**Revocation** is the harder path — Bell's `revoke` is an *event*, and the controller
learns of it by polling (`watch_revoked`, met in 07). So an examiner should object: *"your
revocation lag is just your polling choice."* E9 answers by sweeping the watcher's poll
interval across 0.1–2.0 s (six fresh stacks per setting) and measuring on-chain revoke →
policer-gone at each. Look for: the measured curve hugging the `lag = poll` reference line,
with a floor peeking out at the smallest poll.

In [ ]:
polls = sorted({r["poll_s"] for r in SWEEP})
lag = {p: statistics.median([r["revocation_lag_s"] for r in SWEEP if r["poll_s"] == p]) for p in polls}

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot([p * 1000 for p in polls], [lag[p] * 1000 for p in polls], "o-",
        color=C["alert"], lw=2, ms=8, label="measured lag (median of 6)")
ax.plot([0, 2000], [0, 2000], "--", color=MUTED, lw=1.5, label="lag = poll (reference)")
ax.set_xlabel("watcher poll interval (ms)"); ax.set_ylabel("revocation lag (ms)")
ax.set_title("E9 — revocation lag is the poll dial, not an architectural cost", color=INK, pad=8)
ax.legend(); plt.tight_layout(); plt.show()

for p in polls:
    print(f"  poll {p:4} s → lag {lag[p] * 1000:6.0f} ms")
assert [lag[p] for p in polls] == sorted(lag[p] for p in polls)   # lag rises with the poll…
assert lag[2.0] / lag[0.1] > 5                                    # …and is dominated BY the poll

The honest sentence, then: **revocation lag ≈ poll interval + a ~80 ms actuation floor.**
The poll is an *operator dial* — a cost-vs-reaction-time trade (poll faster, pay more RPC
queries) — while the floor (detection + one gNMI teardown) is the architectural minimum.
The floor is only visible at the smallest poll: at 0.1 s the lag is ~182 ms (poll + ~80 ms);
by 2.0 s it has drowned inside the poll. A *chosen dial, not a defect* — and the boundary
matters: this chain mines instantly and its one node sees events at once. On a public
chain, event visibility itself waits for a block, which adds block time to the floor.

In [ ]:
print(f"{'poll':>8}{'lag':>11}{'lag − poll':>14}")
for p in polls:
    print(f"{p:7.2f}s{lag[p] * 1000:9.0f}ms{(lag[p] - p) * 1000:+12.0f}ms")

floor = lag[0.1] - 0.1
print(f"\nactuation floor (visible at the smallest poll) ≈ {floor * 1000:.0f} ms")
assert 0.05 < floor < 0.12
boundary(1, 3, 4)

**✏️ Your turn 5.1 — the lag/poll ratio.** Divide each measured lag by its poll interval.
Predict first (as the comment): at which poll is the ratio largest, and why there?
Success: both asserts pass — the ratio only leaves ~1.0 where the ~80 ms floor is a big
fraction of the poll itself.

In [ ]:
# My prediction: lag/poll is largest at poll = ____ s, because ____
ratios = {p: lag[p] / p for p in polls}
for p, r in ratios.items():
    print(f"  poll {p:4} s → lag/poll = {r:.2f}")

# TODO: un-comment once you've predicted:
# assert max(ratios, key=ratios.get) == 0.1                        # the floor is ~80% of a 0.1 s poll
# assert all(0.9 < ratios[p] < 1.1 for p in polls if p >= 0.25)    # …and invisible beyond it

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
assert max(ratios, key=ratios.get) == 0.1
assert all(0.9 < ratios[p] < 1.1 for p in polls if p >= 0.25)
```

At poll = 0.1 s the fixed ~80 ms actuation floor nearly doubles the lag (ratio 1.8); from
0.25 s up, the poll dominates and the ratio pins to ~1.0 — which is exactly what
"lag ≈ poll + constant" predicts.
</details>

## 6. E3 — what it costs (gas, then dollars — carefully)

Gas (06) is the EVM's deterministic unit of computational work: the same transaction on the
same contract at the same state costs the same gas — everywhere, forever. The harness read
`gasUsed` off every receipt. One trap here, caught by the audit: `fulfill` is **bimodal** —
telemetry offers carry a much larger ABI-encoded `params` blob (sensor paths, collector
endpoint — you dissected one by hand in 04), and bytes cost gas to store. Pool the two
services and the "median" describes a transaction that never happened. Watch that happen
live below — and notice `approve`/`revoke` have *zero* spread across 40 runs each: that is
what deterministic execution looks like in data.

In [ ]:
fulfill = {svc: sorted(r["gas"]["fulfill"] for r in LAT if r["ok"] and r["service"] == svc)
           for svc in ("bandwidth", "telemetry")}

for svc, g in fulfill.items():
    print(f"  fulfill/{svc:10} median {statistics.median(g):9,.0f} gas   [{g[0]:,}–{g[-1]:,}]  (n={len(g)})")
approve = [r["gas"]["approve"] for r in LAT if r["ok"] and "approve" in r["gas"]]
revoke = [r["gas"]["revoke"] for r in LAT if r["ok"] and "revoke" in r["gas"]]
print(f"  approve            median {statistics.median(approve):9,.0f} gas   identical in all {len(approve)} runs: {set(approve)}")
print(f"  revoke             median {statistics.median(revoke):9,.0f} gas   identical in all {len(revoke)} runs: {set(revoke)}")

assert statistics.median(fulfill["bandwidth"]) == 268_050
assert statistics.median(fulfill["telemetry"]) == 447_371
assert set(approve) == {46_366} and set(revoke) == {29_903}   # gas is DETERMINISTIC — zero spread

pooled = statistics.median(fulfill["bandwidth"] + fulfill["telemetry"])
gap = min(abs(g - pooled) for g in fulfill["bandwidth"] + fulfill["telemetry"])
print(f"\npooled 'median' → {pooled:,.0f} gas — the nearest REAL transaction is {gap:,.0f} gas away")
assert gap > 60_000     # the audit's point, reproduced: a pooled median describes no real tx

One measurement can hide a systematic error; two *independent* measurements agreeing can't
(easily). The Foundry test suite measures the same contract by a completely different route:
`forge snapshot` (06) re-runs every Solidity test and records its gas into
`contracts/.gas-snapshot`. Parse that file live and compare. The snapshot's happy-path
fulfills land at ~322k–348k versus the harness's 268k — same order of magnitude, not the
same number, and the difference is explainable: different offer fixtures, and storage
*warmth* (the first-ever write to a storage slot costs extra; the harness's chain was warm
from prior runs, each Foundry test starts cold). Agreement to the same order is exactly the
confidence a cross-check buys.

In [ ]:
import re

snapshot = (ROOT / "contracts" / ".gas-snapshot").read_text()
happy = [int(g) for g in re.findall(r"test_fulfill\w*\(\) \(gas: (\d+)\)", snapshot)]
fuzz = [int(g) for g in re.findall(r"testFuzz_\w+\([^)]*\) \(runs: \d+, μ: (\d+)", snapshot)
        if int(g) > 100_000]      # drop the cheap revert-path fuzzes; we want completed fulfills

print("forge-snapshot fulfill gas (happy paths):", [f"{g:,}" for g in happy])
print("forge-snapshot fulfill gas (fuzz means): ", [f"{g:,}" for g in fuzz])

harness = statistics.median(fulfill["bandwidth"])
for g in happy + fuzz:
    assert 1.0 < g / harness < 1.5, (g, g / harness)   # same order; fixtures + warmth explain the rest
print(f"✓ two independent measurements agree: harness {harness:,.0f} vs snapshot "
      f"{min(happy + fuzz):,}–{max(happy + fuzz):,} — both ~2.7–3.5 × 10⁵ gas")

Now dollars — and a change of *epistemic register*. Gas is a measurement; a dollar figure
is gas × gas-price × token-price, and those two prices are **scenario parameters, not
measurements** — docs/09 marks its own table "ETH \$3000, illustrative, 2026-07". A *gwei*
is 10⁻⁹ ETH, the unit gas prices are quoted in. Three scenarios: an L2 rollup — where fees
like these actually live, with the honest asterisk that an L2 user *also* pays an L1
data-fee for calldata that was **not** measured here — then calm L1, then busy L1.

In [ ]:
def usd(gas: int, gwei: float, eth_usd: float) -> float:
    return gas * gwei * 1e-9 * eth_usd            # gas × gas-price(in ETH) × ETH price

ETH = 3000                                        # scenario parameter — NOT a measurement
scenarios = [("L2 @ 0.03 gwei", 0.03), ("L1 calm @ 8 gwei", 8), ("L1 busy @ 30 gwei", 30)]

print(f"{'fulfill cost':22}" + "".join(f"{name:>18}" for name, _ in scenarios))
for svc in ("bandwidth", "telemetry"):
    g = statistics.median(fulfill[svc])
    print(f"  {svc:20}" + "".join(f"{'$' + format(usd(g, gwei, ETH), '.3f'):>18}" for _, gwei in scenarios))

assert usd(statistics.median(fulfill["bandwidth"]), 0.03, ETH) < 0.05    # cents on a rollup
assert usd(statistics.median(fulfill["telemetry"]), 30, ETH) > 30       # tens of dollars, busy L1
print("\n→ feasible on any rollup; on L1 the fee SHAPES THE PRODUCT (lease longer windows to")
print("  amortize, don't price per-flow) — a feasibility boundary, not a failure")

**✏️ Your turn 6.1 — find the boundary.** House rule 3: no number without its boundary.
Which of the five simulation boundaries does the **268,050 gas** number carry? Look back at
the `BOUNDARIES` dict, pick 1–5 — or decide none applies — and record your answer. Success:
your un-commented assert passes, and you can say in one sentence what fine print the gas
number *does* carry instead.

In [ ]:
boundary(*BOUNDARIES)                  # the five suspects, reprinted

carries_boundary = 0                   # TODO: set to 1, 2, 3, 4, 5 — or None if none applies

print("your answer →", carries_boundary)
# TODO: un-comment once you've decided:
# assert carries_boundary is None

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
carries_boundary = None
assert carries_boundary is None
```

Trick question — deliberately. Execution gas is deterministic EVM arithmetic, exact at the
pinned compiler (the zero-spread `approve`/`revoke` proved it), so none of the five
*simulation* boundaries caps it. Its fine print is different: the **dollar** translation is
a scenario, and on an L2 the user fee adds an unmeasured L1 data-fee (docs/09 §11, last row).
</details>

## 7. E4 — the adversarial matrix: two layers, twelve rejections

13 showed you the harness's `attempt()` trick: run each attack end-to-end on the real stack
and let the *exception type* attribute the rejection — `ChainRevert` means the contract
refused (bad **money**: forged, replayed, lapsed offers), `Denied` means the controller
refused (bad **access**: wrong owner, burnt nonce, wrong time, wrong scope, revoked
ticket). The two layers are independent — neither needs the other to hold. Each record also
carries the layer the design *predicted* would catch it, so the matrix below checks
prediction against reality: twelve for twelve.

In [ ]:
from collections import Counter

attacks = [r for r in ADV if not r.get("by_design")]
print(f"{'attack':48}{'rejected by':13}{'code'}")
print("-" * 84)
for r in attacks:
    print(f"{r['attack']:48}{r['layer']:13}{r['code']}")

layers = Counter(r["layer"] for r in attacks)
print("-" * 84)
print(f"{sum(r['rejected'] for r in attacks)}/{len(attacks)} rejected → "
      f"{layers['contract']} at the contract (bad money), {layers['controller']} at the controller (bad access)")

assert len(attacks) == 12 and all(r["rejected"] for r in attacks)
assert layers == Counter({"controller": 9, "contract": 3})
assert all(r["layer"] == r["expected_layer"] for r in attacks)   # every attack died at its PREDICTED layer
print("✓ defense in depth — and the design predicted every rejection point")

**✏️ Your turn 7.1 — read one attack record.** In 08 you defeated a replay attack yourself:
a challenge nonce, once used, burns. Find that attack's record in `ADV` and — *before*
printing it — write which layer rejected it and by what code. Success: both asserts pass
without surprising you.

In [ ]:
rec = next(r for r in ADV if "nonce" in r["attack"])
# My prediction: layer = ______ , code = ______

print(rec)

# TODO: un-comment once predicted:
# assert rec["layer"] == "controller"          # replay protection is the bouncer's job…
# assert rec["code"] == "E_NONCE_REUSED"       # …and 08's AuthStore is the check that fired

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
assert rec["layer"] == "controller"
assert rec["code"] == "E_NONCE_REUSED"
```

The chain never sees a challenge nonce — challenges are controller-local state (08), so
only the controller *can* reject this one. Layer attribution is architecture, not chance.
</details>

Honesty about what this matrix is: **enumerated tests of the documented threat model,
written by the system's own author** — every guard fired at its designed layer, and every
rejection was raised *upstream of any gNMI call* (a property of where the checks sit in the
code path, not a per-attack device readback). It is *not* a fuzzing campaign or an economic
adversary. Three things were **not attacked at all**: the chain itself (consensus attacks,
reorgs racing the watcher, front-running `fulfill`), the router OS beneath the controller,
and key custody — a stolen key *is* the owner, which is exactly why rule 2 confines keys to
one process: no layer above can help once a key leaks. And one case was **allowed on
purpose** — read its record:

In [ ]:
by_design = next(r for r in ADV if r.get("by_design"))
print(f"attack : {by_design['attack']}")
print(f"outcome: ALLOWED — {by_design['by_design']}")

untested = ["chain-level adversaries (reorgs, front-running fulfill)", "input fuzzing",
            "router-OS compromise", "key theft / custody failure", "concurrency & load"]
print("\nnot attacked (named as future work, not glossed over):")
for u in untested:
    print("  ✗", u)

assert by_design["rejected"] is False and sum(1 for r in ADV if r.get("by_design")) == 1
print("\n✓ a second valid ticket DOES configure the device — overselling is the provider")
print("  ledger's concern at quote time (M5.2, met in 10), not a controller security check")

## 8. E5 — the judgment layer, reported precisely

The two LLM slots against the deployed Qwen3-4B, one session (boundary 5). Precision in the
*claims* is everything here. The quote slot (Bell prices a need) was graded on **schema
validity**: did the guarded `structured()` call from 10 return a valid `QuoteDecision`
inside the allowed [5, 25] TOK band? Ten needs spanning 10→500 Mbps — a 50× capacity range
— went 10/10 on the first attempt, zero retries. And then the finding this evaluation
*kept* instead of hiding: look at the middle column.

In [ ]:
quotes = [r for r in LLMJ if r["slot"] == "quote"]
print(f"{'need':>10}{'quoted':>10}{'in [5,25]':>11}{'attempts':>10}")
for r in quotes:
    print(f"{r['case']:>10}{r['quoted']:>7} TOK{str(r['in_range']):>9}{r['attempts']:>10}")

assert len(quotes) == 10 and all(r["ok"] and r["in_range"] and r["attempts"] == 1 for r in quotes)
assert {r["quoted"] for r in quotes} == {10}     # ← EVERY quote is the stated list price
print("\n→ 10/10 schema-valid, first attempt — but the model anchored to the list price")
print("  across a 50× capacity range: schema compliance PROVEN, price discovery ABSENT")

The decide slot (Ada accepts or rejects an offer against her budget) was graded against
ground truth — *accept iff the offer meets the need and price ≤ budget* — over 12 curated
cases, three deliberately on the `price == budget` boundary, where "accept at equality" is
a prompt convention rather than a fact of arithmetic, so they are split out. This is a
smoke test of a one-comparison judgment wrapped in a fail-safe guard (invalid JSON → retry
→ safe decline, the loop you drove in 10) — **not** a robustness benchmark: no malformed
offers, one sample per case. Precisely stated, it went twelve for twelve.

In [ ]:
decides = [r for r in LLMJ if r["slot"] == "decide"]
boundary_cases = [r for r in decides if r["boundary"]]

for r in decides[:4]:
    tag = "  ← boundary (price == budget)" if r["boundary"] else ""
    print(f"  {r['case']:8} expected {str(r['expected']):5} got {str(r['got']):5}{tag}")
print(f"  … and {len(decides) - 4} more\n")

nonb = [r for r in decides if not r["boundary"]]
print(f"decide accuracy → {sum(r['correct'] for r in decides)}/{len(decides)} correct "
      f"({sum(r['correct'] for r in nonb)}/{len(nonb)} excluding the {len(boundary_cases)} boundary ties)")

assert sum(r["correct"] for r in decides) == len(decides) == 12 and len(boundary_cases) == 3
assert all(r["attempts"] == 1 for r in decides)   # the fail-safe guard never even had to fire
print("✓ 12/12 against ground truth — clean threshold judgment, single sample per case")

What a negotiation costs. Tokens are the LLM's billing unit (one token ≈ ¾ of an English
word); one negotiation = one quote + one decide. Token counts are a *measurement*; the
dollar figure depends on a per-token price, so it is bracketed between a cheap self-hosted
rate and a mid-tier hosted one — the same scenario-parameter honesty as the gas table.

In [ ]:
tok = lambda r: r["usage"]["prompt_tokens"] + r["usage"]["completion_tokens"]
q_tok = statistics.median([tok(r) for r in quotes])
d_tok = statistics.median([tok(r) for r in decides])
total = q_tok + d_tok

print(f"tokens per negotiation → {total:.0f}  (quote {q_tok:.0f} + decide {d_tok:.0f})")
for rate in (0.20, 2.00):
    print(f"  @ ${rate:.2f}/Mtok → ${total / 1e6 * rate:.6f} per negotiation")

assert total == 1114
assert round(total / 1e6 * 0.20, 6) == SUMMARY["llm"]["cost_per_negotiation"]["usd_at_0.20_per_Mtok"]
print("\n→ judgment is viable INSIDE its fence: accurate on clean cases, fail-safe, a fraction")
print("  of a cent — while negotiation STRATEGY (price discovery) was NOT evaluated, and the")
print("  report says so in bold instead of rounding it up to a win")
boundary(5)

**✏️ Your turn 8.1 — re-price the negotiation.** The bracket above tops out at \$2/Mtok.
Price the same 1114-token negotiation at a frontier-model rate of \$15/Mtok and see whether
the "negligible cost" conclusion survives a 7.5× price hike. Success: still under two cents
— conclusions that survive their scenario parameters moving are the ones worth keeping.

In [ ]:
FRONTIER = 15.00                       # $/Mtok — an expensive frontier model, ~top of market
my_cost = 0.0                          # TODO: price the `total`-token negotiation at FRONTIER

print(f"negotiation at ${FRONTIER:.2f}/Mtok → ${my_cost:.4f}")

# TODO: un-comment once computed:
# assert my_cost < 0.02                # under 2¢ — the conclusion survives the model choice

<details><summary>✅ Solution 8.1 — peek only after trying</summary>

```python
my_cost = total / 1e6 * FRONTIER      # ≈ $0.0167
assert my_cost < 0.02
```

Even at frontier prices a negotiation costs well under a plausible service price — the
per-negotiation LLM cost is noise next to the 10 TOK deal it brokers.
</details>

## 9. E6 — the price of trustlessness

The last cost question is the sharpest: what does all this machinery *add* over just…
configuring the router? E6 measured the bare alternative — one direct `netctl`
`apply_bandwidth` call (09), no agents, no chain, no controller, no signatures — on the
same 50 Mbps path, twenty times. Everything the thesis adds is the *difference* between
that and section 3's deterministic lifecycle.

**✏️ Your turn 9.1 — predict the overhead first.** You already know both inputs: the bare
write is ~20 ms, the full det lifecycle ~89 ms. Write your predicted overhead in ms as the
comment *before* running the subtraction. Success: your prediction lands within a few ms —
and you can guess what the overhead is mostly made of before the next cell names it.

In [ ]:
bare = spread([r["apply_s"] for r in BASE if r["ok"]])
full = spread(det_e2e)
show_ms("bare netctl apply (no trust machinery)", bare)
show_ms("full det lifecycle (request→enforced)", full)

# My prediction: trustlessness adds ≈ ____ ms
overhead = 0.0                        # TODO: full median − bare median (both in seconds)
print(f"\ntrustlessness overhead → {overhead * 1000:.1f} ms")

# TODO: un-comment once computed:
# assert 0.055 < overhead < 0.085     # ~69 ms

<details><summary>✅ Solution 9.1 — peek only after trying</summary>

```python
overhead = full["median"] - bare["median"]     # ≈ 0.069 s
assert 0.055 < overhead < 0.085
```

~69 ms. If you predicted "89 − 20", you did exactly what the report does — the subtraction
IS the experiment; the twenty bare runs exist so the 20 ms is a median, not a guess.
</details>

Decompose the ~69 ms and it is almost entirely *settlement and signatures* — the predicate
itself, section 4 measured, contributes 0.0001 ms of it. What the milliseconds buy:
**atomic settlement** (payment and entitlement cannot split — 06's invariants), **no shared
credentials** (Ada never holds router access, only a ticket), **an audit trail** (every
grant is a public transaction), and **revocability** (section 5's kill switch). The 20 ms
gNMI write was never the product; the other ~69 ms *is* the entire trust story — and even
that is an instant-mine lower bound.

In [ ]:
det_rows = [r for r in LAT if r["mode"] == "det" and r["ok"]]
part = lambda f: statistics.median([f(r["phases"]) for r in det_rows]) * 1000
parts = {
    "settle (the on-chain write)": part(lambda p: p["settle_s"]),
    "sign offer + sign proof": part(lambda p: p["sign_offer_s"] + p["sign_proof_s"]),
    "challenge": part(lambda p: p["challenge_s"]),
    "controller compute + chain reads": part(lambda p: p["activate_s"] - p["gnmi_apply_s"]),
}
bare_ms = statistics.median([r["apply_s"] for r in BASE if r["ok"]]) * 1000
overhead_ms = statistics.median(det_e2e) * 1000 - bare_ms

for name, v in parts.items():
    print(f"  {name:34} {v:5.1f} ms")
print(f"  {'≈ overhead (full − bare)':34} {overhead_ms:5.1f} ms   ← parts ≈ whole (medians don't add exactly)")
assert abs(sum(parts.values()) - overhead_ms) < 15

fig, ax = plt.subplots(figsize=(8.5, 1.9))
ax.barh([0], [bare_ms], color=MUTED, edgecolor="white", height=0.5,
        label=f"bare device write ({bare_ms:.0f} ms)")
ax.barh([0], [overhead_ms], left=bare_ms, color=C["controller"], edgecolor="white", height=0.5,
        label=f"+ trust machinery ({overhead_ms:.0f} ms — predicate itself <0.0001 ms)")
ax.set_yticks([]); ax.set_xlabel("milliseconds")
ax.set_title("E6 — what trust-minimization adds to a bare write", color=INK, pad=8)
ax.legend(loc="lower right", fontsize=9, framealpha=1)
plt.tight_layout(); plt.show()
boundary(1, 2)

## 10. Threats to validity — which numbers survive?

The examiner's last move: *"lovely numbers — which of them are true anywhere but your
laptop?"* docs/09 §11 consolidates the answer; here it is walked number by number. The
pattern to internalize: numbers made of **deterministic computation** (EVM gas, the pure
predicate, one device round-trip) travel unchanged; numbers containing a **wait** (block
time, transport, polling) are lower bounds that must be remeasured; numbers containing a
**price** were never measurements at all.

In [ ]:
verdicts = [
    # (number, boundaries it carries, what happens on a public testnet)
    ("predicate 86 ns", [4], "SURVIVES — pure CPU; no chain, no transport inside it"),
    ("gas 268k / 447k", [], "SURVIVES — deterministic EVM execution (the FEE doesn't)"),
    ("actuation floor ~80 ms · expiry 73 ms", [3, 4], "SURVIVES — one gNMI round-trip to a real device"),
    ("settle 38 ms", [1], "REMEASURE — add block time (~2 s L2, ~12 s L1)"),
    ("e2e 89 ms / 3.27 s", [1, 2, 3, 4], "REMEASURE — add transport hops + block time"),
    ("revocation lag ≈ poll", [1], "REMEASURE — event visibility itself waits a block"),
    ("LLM 12/12 · 1114 tok · ~1.6 s", [5], "THIS DEPLOYMENT — tokens travel, latency doesn't"),
    ("$0.024 / $6–40", [], "SCENARIO — a price assumption, never a measurement"),
]
for number, bs, verdict in verdicts:
    tag = ",".join(str(b) for b in bs) or "—"
    print(f"  {number:40} boundaries [{tag:8}] {verdict}")

survivors = [n for n, _, v in verdicts if v.startswith("SURVIVES")]
assert len(survivors) == 3
print(f"\n✓ {len(survivors)} of 8 rows survive a public testnet untouched — and they are exactly")
print("  the rows this thesis CONTRIBUTES: the predicate, the contract, the actuation path")

**✏️ Your turn 10.1 — classify four numbers yourself.** For each number below, decide:
would it survive a move to a public testnet unchanged (`True`), or not (`False`)? Use the
pattern — deterministic computation travels; waits and prices don't. Success: the assert
passes and you can justify each answer in one clause.

In [ ]:
survives = {                          # TODO: replace each None with True or False
    "86 ns predicate": None,
    "89 ms det e2e": None,
    "268k gas fulfill": None,
    "$0.024 L2 fulfill": None,
}
print(survives)

# TODO: un-comment when filled in:
# assert survives == {"86 ns predicate": True, "89 ms det e2e": False,
#                     "268k gas fulfill": True, "$0.024 L2 fulfill": False}

<details><summary>✅ Solution 10.1 — peek only after trying</summary>

```python
survives = {"86 ns predicate": True,      # pure CPU
            "89 ms det e2e": False,       # contains waits (chain, transport) — a lower bound
            "268k gas fulfill": True,     # deterministic EVM execution
            "$0.024 L2 fulfill": False}   # contains prices — a scenario, not a measurement
```

Computation travels; waits and prices don't. That one sentence is most of §11.
</details>

## 11. Conclusions — each pinned to its evidence

A conclusion you can't pin to a file and a number is an opinion. Five claims survived this
chapter, printed below with their evidence pins. Claim (a) deserves its extra beat: *two
genuinely different services* — a rate limiter and a telemetry subscription — flowed through
**identical settlement and authorization machinery**; only the translator differed (08).
The data shows it twice at once: their lifecycle medians agree to a fraction of a
millisecond, while their gas differs by ~180k — same machine, different payload.

In [ ]:
e2e_by_svc = {svc: statistics.median([r["phases"]["e2e_request_to_enforced_s"] for r in LAT
              if r["mode"] == "det" and r["service"] == svc and r["ok"]])
              for svc in ("bandwidth", "telemetry")}
assert abs(e2e_by_svc["bandwidth"] - e2e_by_svc["telemetry"]) < 0.005            # same machinery…
assert statistics.median(fulfill["telemetry"]) - statistics.median(fulfill["bandwidth"]) > 150_000  # …different payload

pins = [
    ("a", "the settlement pattern is SERVICE-AGNOSTIC — det e2e "
          f"{e2e_by_svc['bandwidth'] * 1000:.1f} ms (bw) vs {e2e_by_svc['telemetry'] * 1000:.1f} ms (tel), "
          "only the translator differed", "latency.jsonl · E1+E3"),
    ("b", f"deterministic authorization costs NANOSECONDS — {ns['allow']:.0f} ns allow; "
          "the trust boundary adds no meaningful latency", "predicate.jsonl · E7"),
    ("c", f"the token GENUINELY GOVERNS THE WIRE — expiry {statistics.median(ticks) * 1000:.0f} ms, "
          f"revocation lag ≈ poll + {floor * 1000:.0f} ms floor (a tunable dial)",
          "expiry.jsonl + revlag_sweep.jsonl · E2+E9"),
    ("d", f"the cost is QUANTIFIED AND SMALL — {statistics.median(fulfill['bandwidth']):,.0f}/"
          f"{statistics.median(fulfill['telemetry']):,.0f} gas, +{overhead_ms:.0f} ms over a bare write",
          "latency.jsonl + baseline.jsonl · E3+E6"),
    ("e", "LLM judgment is VIABLE INSIDE ITS FENCE and the fence held — 12/12 decisions, "
          f"12/12 attacks rejected, {total:.0f} tok (<1¢)/negotiation", "llm.jsonl + adversarial.jsonl · E5+E4"),
]
for tag, claim, pin in pins:
    print(f" ({tag}) {claim}")
    print(f"     evidence: {pin}\n")

Now the verdict — derived from the data live, never typed in. Each line below is one of
docs/09 §12's six conclusions; each pulls its number from `SUMMARY` or the raw rows, so if
the dataset changed, the verdict would change with it. This is the cell
`evaluation_explore` renders compactly — here, every line also says *why* its number
supports the claim and carries its caveat inline.

In [ ]:
rev05 = statistics.median([r["phases"]["revocation_lag_s"] for r in LAT
                           if r["ok"] and "revocation_lag_s" in r["phases"]]) * 1000
assert 400 < rev05 < 520                          # pooled n=80 at the default 0.5 s poll

print("FEASIBILITY VERDICT — feasible for tokenized provisioning at window/lease timescales\n")
print(f" 1. request→enforced in {statistics.median(det_e2e) * 1000:.0f} ms (det) / "
      f"{statistics.median(llm_e2e):.1f} s (llm); {SUMMARY['runs_ok']}/{SUMMARY['runs_total']} "
      f"lifecycles, {len(SUMMARY['failures'])} failures")
print("      — boringly reliable mechanics; a transport-free, instant-mine lower bound (b. 1, 2, 4)")
print(f" 2. the predicate costs {SUMMARY['predicate']['allow']} ns; trustlessness adds "
      f"~{overhead_ms:.0f} ms over a bare write")
print("      — the security LOGIC is free; the cost is settlement + signatures + chain reads")
print(f" 3. revocation enforced in ~{rev05:.0f} ms at the 0.5 s poll, scaling with the poll "
      f"(a dial + ~{floor * 1000:.0f} ms floor)")
print("      — the ticket is authorization, not paperwork (b. 1: add block time on a public chain)")
print(f" 4. {sum(r['rejected'] for r in ADV if not r.get('by_design'))}/12 attacks rejected at "
      "their predicted layer (3 contract / 9 controller)")
print("      — within the documented threat model; fuzzing / reorgs / economics untested")
print(f" 5. fulfill {SUMMARY['gas']['bandwidth']['fulfill']['median']:,.0f} gas (bw) / "
      f"{SUMMARY['gas']['telemetry']['fulfill']['median']:,.0f} (tel) — cents on an L2, $6–40 on L1")
print("      — feasible on any rollup; on L1 the fee shapes the product")
print(f" 6. judgment 12/12 correct at ~{SUMMARY['llm']['cost_per_negotiation']['total_tokens_median']:.0f} "
      "tokens (<1¢) per negotiation")
print("      — an agent market is viable (b. 5); price DISCOVERY not demonstrated")

**✏️ Your turn 11.1 — add the missing line.** The verdict never mentions expiry. Write
line 7 yourself: pull the expiry median from `SUMMARY` (key `expiry_tick_to_deconfig_s`),
convert to milliseconds, and give it a one-line caveat the way lines 1–6 do. Success: the
assert passes and your caveat says what the ~73 ms is physically made of.

In [ ]:
expiry_ms = 0.0     # TODO: SUMMARY["expiry_tick_to_deconfig_s"]["median"] * 1000

print(f" 7. chain-time expiry becomes device deconfiguration in ~{expiry_ms:.0f} ms")
print("      — TODO: your caveat (what is it made of? which boundary caps it?)")

# TODO: un-comment once pulled:
# assert 65 < expiry_ms < 85

<details><summary>✅ Solution 11.1 — peek only after trying</summary>

```python
expiry_ms = SUMMARY["expiry_tick_to_deconfig_s"]["median"] * 1000
assert 65 < expiry_ms < 85
# caveat: one synchronous gNMI delete — a device round-trip (b. 3, 4); no tx, no watcher,
# so block time does NOT join this one on a public chain
```

Expiry is the rare wire-governance number that mostly *survives* the move to a public
chain: the deadline is already on the ticket, so no event needs to be seen.
</details>

## 12. Future work — scoped from the negative findings

An honest evaluation's negative findings *are* its future-work section. Nothing speculative
belongs here: each item below traces to a gap you computed yourself in this chapter, which
is what makes it scoped work rather than hand-waving.

In [ ]:
future = {
    "quotes anchored to the list price (§8)":
        "price discovery — richer market context, multi-round negotiation; grade STRATEGY, not schema",
    "instant-mine, single-node chain (boundary 1)":
        "repeat E1/E2 on a public L2 testnet — remeasure settle + revocation lag under real block times",
    "ADR-006 datapath carve-out (boundary 3)":
        "hardware datapath enforcement — QoS-enforcing silicon; packet-level proof, not config readback",
    "enumerated threat model, one machine (§7, boundary 4)":
        "adversarial conditions — fuzzing, reorgs / front-running, concurrency and load (no throughput claims yet)",
}
for finding, work in future.items():
    print(f"  finding: {finding}")
    print(f"     next: {work}\n")
print(f"✓ {len(future)} negative findings → {len(future)} scoped experiments — nothing hidden, nothing vague")

## 13. The real final exam — four questions, in writing

Notebook 12's exam checked the *system*; this one checks the *evaluation*. These are the
four explain-back questions from [`docs/evidence/M7.1.md`](../../../docs/evidence/M7.1.md)
— the ones the author had to answer before the defense. **Write your answers down** (really
write them — prose exposes gaps that nodding hides), then unfold each model answer.

**Q1.** Why is the predicate microbenchmark (86 ns) a *stronger* claim than "activate takes
38 ms" — and what exactly does the 38 ms bundle that the 86 ns excludes?
<details><summary>Model answer</summary>

`activate` bundles chain reads (ownership + revocation via the reader), session
bookkeeping, and the gNMI Set around the decision. The microbenchmark isolates the
security-critical judgment itself, so "the trust boundary adds no meaningful latency" is
*measured*, not inferred from a bundle where I/O dominates. And the 86 ns is pure CPU — it
travels to any deployment; the 38 ms varies with chain and device.
</details>

**Q2.** The sweep says lag ≈ poll + ~80 ms. Which term is an architectural property, and
what changes on a public chain?
<details><summary>Model answer</summary>

The ~80 ms floor is architectural — detection plus one gNMI teardown, a device round-trip
no dial removes. The poll is an operator SLO knob (cost vs reaction time). On a public
chain, *event visibility* itself waits for a block, so block time joins the floor: the dial
stays a dial, but its zero point moves.
</details>

**Q3.** Why must fulfill gas be reported per service type — what makes telemetry 447k vs
bandwidth 268k?
<details><summary>Model answer</summary>

Telemetry offers carry a much larger ABI-encoded `params` blob (sensor paths, collector
endpoint, interval) that the contract stores verbatim at mint; calldata and storage cost
gas per byte. The distribution is bimodal, so a pooled median (~383k) describes no
transaction that ever happened — you reproduced that trap live in section 6.
</details>

**Q4.** The second-ticket-on-same-resource "attack" is allowed. Where is overselling
actually prevented, and why is that the right layer?
<details><summary>Model answer</summary>

At the provider's `CapacityLedger`, at quote time (M5.2 — met in 10): capacity is the
seller's inventory decision, made *before* money moves. The controller's `E_CONFLICT` is
per-ticket security — one entitlement, one active session. Folding inventory into the
controller would couple the security kernel to business state; the bouncer checks tickets,
it doesn't decide how many the club sells.
</details>

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| poisoned a mean with one straggler | median + `[min–max]` + `n` — the honest triple | every number you ever report |
| attached `boundary(…)` to each result | no number without its simulation boundary | docs/09 §2 + §11, the written form |
| recomputed 89 ms / 3.27 s and matched `summary.json` bit-for-bit | raw JSONL is the evidence; the summary is only the claim | reproducing the campaign (docs/09 §13) |
| set 86 ns against a 1.6 s judgment slot | purity → auditable AND computationally free | rule 4's payoff, now measured |
| plotted lag ≈ poll + ~80 ms | an operator dial vs an architectural floor | revocation SLOs on a real deployment |
| reproduced the 383k pooled-median trap | a statistic must describe a real transaction | the audit story in M7.1 |
| cross-checked harness gas against `forge snapshot` | two independent measurements agreeing = confidence | any measurement you ever trust |
| priced a negotiation at <1¢ and a fulfill at \$0.024–40 | measurements vs scenario parameters | the L2-vs-L1 product shaping |
| derived the verdict live from the dataset | conclusions pinned to files and numbers | defending this thesis — or your own |

## Experiments to try

Predict first, then run:

- **Your own nanoseconds:** rerun the predicate microbenchmark on your machine —
  `uv run python -m e2e.experiments --exp predicate --out /tmp/eval` (lab-free, 13's
  recipe; mind the `--out`: never point it at the committed `e2e/runs/eval`). Predict which
  changes versus the committed data: the ns magnitudes, or the denial *order*? (Order is
  code structure; magnitude is hardware.)
- **Deliberate breakage — resurrect the audited-out bug:** compute a nearest-rank p95 with
  `int(0.95 * n)` instead of `math.ceil(0.95 * n) - 1` on any 20-sample phase list, and
  watch the "p95" equal the max. Then read `_stats`'s docstring in
  `e2e/src/e2e/experiments.py` — the audit catch, made runnable.
- **Move the scenario:** re-price section 6 at today's real gas and ETH prices. Predict
  whether any *conclusion* changes before you look. (That invariance is what "scenario
  parameter" means.)
- **The wrong overhead:** compute `llm e2e − bare write` (~3.25 s) and explain why that is
  NOT the price of trustlessness — which rule makes the LLM share a swappable policy
  choice rather than trust machinery?

## Check yourself

1. Why medians and not means for latency-like data — and what two things must always ride
   along with a median?
2. Which headline numbers survive a move to a public testnet unchanged, and what property
   do they share?
3. What did the quote slot's 10/10 actually validate — and what claim on that same data
   would be an overclaim?
4. The pooled fulfill median is ~383k gas. Why does docs/09 refuse to print it?
5. Where is overselling prevented, and why is the controller the wrong layer for it?

## Where this goes next

This is the course's last chapter — the story you met in 00 is now a measured, bounded,
defended system. Two roads onward: **outward** — bring up the lab and rerun the campaign
(docs/09 §13), or point the stack at a public testnet and remeasure section 10's REMEASURE
rows; **inward** — open the blank bench
([`scratch_inspect.ipynb`](../scratch_inspect.ipynb)) and ask the repo your own questions.

## Deeper dive

- [`evaluation_explore.ipynb`](../evaluation_explore.ipynb) — the compact five-figure set
  this chapter taught line by line
- [`docs/09-evaluation.md`](../../../docs/09-evaluation.md) — the written report; §11 is
  the consolidated threats-to-validity table you now know how to read
- [`docs/evidence/M7.1.md`](../../../docs/evidence/M7.1.md) — the evidence file, including
  the adversarial audit that forced the corrections this chapter celebrated